In [1]:
# IMPORTAR LIBRERÍAS NECESARIAS
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import scipy.io as sio
from scipy.signal import butter, sosfiltfilt, hilbert, spectrogram
from scipy.interpolate import interp1d
import h5py
import pandas as pd

In [2]:
# 1. PARÁMETROS DEL REGISTRO
n_channels = 16
fs = 2500  # Hz

# Filtro Butterworth (100-300 Hz)
f_low, f_high = 100, 300
sos_ripple = butter(4, [f_low, f_high], btype="band", fs=fs, output="sos")

# Parámetros de la ventana (30 ms a cada lado para visualizar bien la dinámica)
ventana_ms = 30
puntos_ventana = int((ventana_ms / 1000) * fs)  
tiempo = np.linspace(-ventana_ms, ventana_ms, (puntos_ventana * 2) + 1) / 1000  # en segundos (-0.05 a 0.05)

# Ruta principal
carpeta_principal = Path(r"D:\Violeta\ANALISIS")
archivos_dat = list(carpeta_principal.rglob("LFP_downsampled.dat"))

In [3]:
# 2. LOCALIZAR Y COMPROBAR TODAS LAS SESIONES
# Carpeta principal donde están todos los grupos y sesiones
carpeta_principal = Path(r"D:\Violeta\ANALISIS")

# Buscar todos los archivos LFP_downsampled.dat dentro de ANALISIS
archivos_lfp = sorted(carpeta_principal.rglob("LFP_downsampled.dat"))

# Aquí guardaremos únicamente las sesiones que tengan
# TODOS los archivos necesarios para nuestro análisis
sesiones_validas = []

# Aquí guardaremos las sesiones que tengan algún archivo necesario
# pero les falte otro
sesiones_incompletas = []

# Revisamos una por una las sesiones encontradas
for archivo_lfp in archivos_lfp:

    # La carpeta que contiene LFP_downsampled.dat
    carpeta_sesion = archivo_lfp.parent

    # Archivos que necesitamos en esa misma sesión
    archivo_csd = carpeta_sesion / "rippleCSDs.mat"
    archivo_ripple_analysis = (
        carpeta_sesion / "analyset" / "rippleAnalysis.mat"
    )

    # Comprobar que existen los tres archivos
    tiene_lfp = archivo_lfp.exists()
    tiene_csd = archivo_csd.exists()
    tiene_ripple = archivo_ripple_analysis.exists()

    if tiene_lfp and tiene_csd and tiene_ripple:
        # Guardamos toda la información importante de la sesión
        sesiones_validas.append({
            "grupo": carpeta_sesion.parent.name,
            "sesion": carpeta_sesion.name,
            "carpeta": carpeta_sesion,
            "lfp": archivo_lfp,
            "csd": archivo_csd,
            "ripple_analysis": archivo_ripple_analysis
        })

    else:
        # Guardamos las sesiones incompletas para poder revisarlas
        sesiones_incompletas.append({
            "grupo": carpeta_sesion.parent.name,
            "sesion": carpeta_sesion.name,
            "carpeta": carpeta_sesion,
            "lfp": archivo_lfp,
            "tiene_csd": tiene_csd,
            "tiene_ripple": tiene_ripple
        })

# Ordenar las sesiones por grupo y después por nombre de sesión
sesiones_validas = sorted(
    sesiones_validas,
    key=lambda x: (x["grupo"], x["sesion"])
)
sesiones_incompletas = sorted(
    sesiones_incompletas,
    key=lambda x: (x["grupo"], x["sesion"])
)

# Mostrar un resumen
print("=" * 70)
print("BÚSQUEDA DE SESIONES")
print("=" * 70)

print(f"\nCarpeta principal:")
print(carpeta_principal)

print(f"\nArchivos LFP encontrados: {len(archivos_lfp)}")
print(f"Sesiones válidas: {len(sesiones_validas)}")

# Contar cuántas sesiones hay de cada grupo
print("\n" + "-" * 70)
print("SESIONES VÁLIDAS POR GRUPO")
print("-" * 70)

grupos = sorted(set(s["grupo"] for s in sesiones_validas))

for grupo in grupos:
    numero_sesiones = sum(
        s["grupo"] == grupo
        for s in sesiones_validas
    )
    print(f"{grupo}: {numero_sesiones} sesiones")

print("\n" + "=" * 70)


BÚSQUEDA DE SESIONES

Carpeta principal:
D:\Violeta\ANALISIS

Archivos LFP encontrados: 164
Sesiones válidas: 93

----------------------------------------------------------------------
SESIONES VÁLIDAS POR GRUPO
----------------------------------------------------------------------
EFV_218: 7 sesiones
EFV_220: 8 sesiones
EFV_263: 2 sesiones
EFV_277: 7 sesiones
EFV_279: 6 sesiones
EFV_280: 8 sesiones
NPC_249: 9 sesiones
NPC_250: 9 sesiones
NPC_262: 13 sesiones
NPC_28: 3 sesiones
NPC_29: 3 sesiones
NPC_30: 3 sesiones
NPC_47: 2 sesiones
NPC_48: 3 sesiones
WT_23: 3 sesiones
WT_35: 3 sesiones
WT_90: 4 sesiones



In [4]:
# 3. LEER ARCHIVOS DE DATOS Y CARGAR EL LFP PARA TODAS LAS SESIONES

# Diccionario donde guardaremos las matrices LFP procesadas de cada sesión
datos_lfp_por_sesion = {}
canales_defectuosos_por_sesion = {}

print(f"Cargando LFP para {len(sesiones_validas)} sesiones válidas...\n")

for s in sesiones_validas:
    # Identificador único para la sesión (ej. EFV_218_Sesion1)
    id_sesion = f"{s['grupo']}_{s['sesion']}"
    archivo_lfp_dat = s["lfp"]

    # Leer los datos en bruto en int16
    datos_brutos = np.fromfile(archivo_lfp_dat, dtype=np.int16)

    # Corregir muestras sobrantes si no es divisible entre n_channels
    sobrantes = len(datos_brutos) % n_channels
    if sobrantes != 0:
        datos_brutos = datos_brutos[:-sobrantes]

    # Redimensionar a la matriz (-1 filas x n_channels columnas)
    datos_matriz = datos_brutos.reshape(-1, n_channels)

    # Chequeo básico de calidad por canal (desviación estándar)
    std_canales = np.std(datos_matriz, axis=0)

    # Consideramos sospechoso si std es casi 0 (muerto) o > 3 veces la mediana (ruido masivo)
    mediana_std = np.median(std_canales)
    bad_channels = np.where(
        (std_canales < 1.0) | (std_canales > 3.0 * mediana_std)
    )[0].tolist()

    # Guardar en el diccionario con el ID de la sesión
    datos_lfp_por_sesion[id_sesion] = datos_matriz
    canales_defectuosos_por_sesion[id_sesion] = bad_channels

    alerta = f" ⚠️ Bad channels detectados: {bad_channels}" if bad_channels else ""
    print(f"✓ [{id_sesion}] LFP cargado.{alerta}")

print("\n¡Éxito! Todos los datos de LFP han sido cargados.")

Cargando LFP para 93 sesiones válidas...

✓ [EFV_218_2026-01-12_18-02-06] LFP cargado.
✓ [EFV_218_2026-01-13_12-08-49] LFP cargado.
✓ [EFV_218_2026-01-13_13-09-16] LFP cargado.
✓ [EFV_218_2026-01-15_11-14-47] LFP cargado.
✓ [EFV_218_2026-01-15_11-56-40] LFP cargado.
✓ [EFV_218_2026-01-15_12-33-45] LFP cargado.
✓ [EFV_218_2026-01-16_17-50-35] LFP cargado.
✓ [EFV_220_2026-01-12_16-58-15] LFP cargado.
✓ [EFV_220_2026-01-13_14-12-19] LFP cargado.
✓ [EFV_220_2026-01-13_15-18-23] LFP cargado.
✓ [EFV_220_2026-01-14_11-44-04] LFP cargado.
✓ [EFV_220_2026-01-14_12-37-30] LFP cargado.
✓ [EFV_220_2026-01-15_17-57-21] LFP cargado.
✓ [EFV_220_2026-01-15_18-37-42] LFP cargado.
✓ [EFV_220_2026-01-16_16-53-48] LFP cargado.
✓ [EFV_263_2026-02-10_15-35-14] LFP cargado.
✓ [EFV_263_2026-02-10_16-14-52] LFP cargado.
✓ [EFV_277_2026-02-16_17-04-40] LFP cargado.
✓ [EFV_277_2026-02-17_14-36-39] LFP cargado.
✓ [EFV_277_2026-02-17_15-19-53] LFP cargado.
✓ [EFV_277_2026-02-18_14-28-50] LFP cargado.
✓ [EFV_277_20

In [5]:
# 4. CREAR CARPETAS DE RESULTADOS (INDIVIDUALES Y GLOBAL)

# Crear carpeta global en la carpeta principal 
carpeta_resultados_global = carpeta_principal / "ProcesamientoNatalia_antiguo"
carpeta_resultados_global.mkdir(exist_ok=True)

# Diccionario para guardar las rutas de las carpetas de cada sesión por si las necesitamos después
carpetas_resultados_por_sesion = {}

print(f"Carpeta global de resultados lista en:\n{carpeta_resultados_global}\n")
print(
    f"Creando carpetas de resultados para {len(sesiones_validas)} sesiones...\n"
)

# Crear carpeta de resultados en cada sesión
for s in sesiones_validas:
    id_sesion = f"{s['grupo']}_{s['sesion']}"
    carpeta_sesion = s["carpeta"]

    # Definir ruta y crear carpeta físicamente
    carpeta_resultados = carpeta_sesion / "ProcesamientoNatalia_antiguo"
    carpeta_resultados.mkdir(exist_ok=True)

    # Guardar la ruta en el diccionario
    carpetas_resultados_por_sesion[id_sesion] = carpeta_resultados

    print(f"✓ [{id_sesion}] Carpeta lista: {carpeta_resultados}")

print(
    "\n¡Éxito! Todas las carpetas de resultados han sido creadas correctamente."
)

Carpeta global de resultados lista en:
D:\Violeta\ANALISIS\ProcesamientoNatalia_antiguo

Creando carpetas de resultados para 93 sesiones...

✓ [EFV_218_2026-01-12_18-02-06] Carpeta lista: D:\Violeta\ANALISIS\EFV_218\2026-01-12_18-02-06\ProcesamientoNatalia_antiguo
✓ [EFV_218_2026-01-13_12-08-49] Carpeta lista: D:\Violeta\ANALISIS\EFV_218\2026-01-13_12-08-49\ProcesamientoNatalia_antiguo
✓ [EFV_218_2026-01-13_13-09-16] Carpeta lista: D:\Violeta\ANALISIS\EFV_218\2026-01-13_13-09-16\ProcesamientoNatalia_antiguo
✓ [EFV_218_2026-01-15_11-14-47] Carpeta lista: D:\Violeta\ANALISIS\EFV_218\2026-01-15_11-14-47\ProcesamientoNatalia_antiguo
✓ [EFV_218_2026-01-15_11-56-40] Carpeta lista: D:\Violeta\ANALISIS\EFV_218\2026-01-15_11-56-40\ProcesamientoNatalia_antiguo
✓ [EFV_218_2026-01-15_12-33-45] Carpeta lista: D:\Violeta\ANALISIS\EFV_218\2026-01-15_12-33-45\ProcesamientoNatalia_antiguo
✓ [EFV_218_2026-01-16_17-50-35] Carpeta lista: D:\Violeta\ANALISIS\EFV_218\2026-01-16_17-50-35\ProcesamientoNatalia

In [6]:
# 5. SELECCIÓN DE LOS RIPPLE DETECTADOS PARA TODAS LAS SESIONES

# Diccionario donde guardaremos los índices de los ripples (base 0) de cada sesión
ripples_por_sesion = {}

print(f"Extrayendo ripples detectados de {len(sesiones_validas)} sesiones...\n")

for s in sesiones_validas:
    id_sesion = f"{s['grupo']}_{s['sesion']}"
    archivo_ripple_analysis = s["ripple_analysis"]

    try:
        # Cargar el archivo .mat usando h5py
        with h5py.File(archivo_ripple_analysis, "r") as f:
            # Acceder a rippleAnalysis -> iRipS y transponer
            irips_matlab = f["rippleAnalysis"]["iRipS"][:].T

        # Convertir a enteros y restar 1 para ajustar el índice a Python (base 0)
        irips_python = irips_matlab.astype(int) - 1

        # Guardar en el diccionario
        ripples_por_sesion[id_sesion] = irips_python

        print(
            f"✓ [{id_sesion}] Éxito: {len(irips_python)} ripples detectados."
        )

    except Exception as e:
        print(f"❌ [{id_sesion}] Error al leer {archivo_ripple_analysis.name}: {e}")

print("\n¡Lectura de ripples completada para todas las sesiones!")

Extrayendo ripples detectados de 93 sesiones...

✓ [EFV_218_2026-01-12_18-02-06] Éxito: 67 ripples detectados.
✓ [EFV_218_2026-01-13_12-08-49] Éxito: 25 ripples detectados.
✓ [EFV_218_2026-01-13_13-09-16] Éxito: 44 ripples detectados.
✓ [EFV_218_2026-01-15_11-14-47] Éxito: 30 ripples detectados.
✓ [EFV_218_2026-01-15_11-56-40] Éxito: 25 ripples detectados.
✓ [EFV_218_2026-01-15_12-33-45] Éxito: 66 ripples detectados.
✓ [EFV_218_2026-01-16_17-50-35] Éxito: 40 ripples detectados.
✓ [EFV_220_2026-01-12_16-58-15] Éxito: 105 ripples detectados.
✓ [EFV_220_2026-01-13_14-12-19] Éxito: 35 ripples detectados.
✓ [EFV_220_2026-01-13_15-18-23] Éxito: 29 ripples detectados.
✓ [EFV_220_2026-01-14_11-44-04] Éxito: 104 ripples detectados.
✓ [EFV_220_2026-01-14_12-37-30] Éxito: 90 ripples detectados.
✓ [EFV_220_2026-01-15_17-57-21] Éxito: 32 ripples detectados.
✓ [EFV_220_2026-01-15_18-37-42] Éxito: 88 ripples detectados.
✓ [EFV_220_2026-01-16_16-53-48] Éxito: 13 ripples detectados.
✓ [EFV_263_2026-02-

In [8]:
# 6. CÁLCULO DE AMPLITUD MÁXIMA DE RIPPLE POR CANAL Y RIPPLE MEDIO PARA TODAS LAS SESIONES

matrices_ripples_por_sesion = {}
ripples_medios_por_sesion = {}
amplitudes_por_sesion = {}

print(
    f"Procesando y calculando métricas de ripples para {len(sesiones_validas)} sesiones...\n"
)

for s in sesiones_validas:
    id_sesion = f"{s['grupo']}_{s['sesion']}"

    # Recuperar datos de las celdas anteriores
    datos_matriz = datos_lfp_por_sesion[id_sesion]
    irips_python = ripples_por_sesion[id_sesion]
    carpeta_resultados = carpetas_resultados_por_sesion[id_sesion]
    puntos_de_tiempo = datos_matriz.shape[0]

    # Si la sesión no tiene ripples detectados, saltamos
    if len(irips_python) == 0:
        print(f"⚠️ [{id_sesion}] Sin ripples detectados. Se omite.")
        continue

    # 1. Aplicar filtro para aislar banda ripple (100-300 Hz)
    datos_filtrados = sosfiltfilt(sos_ripple, datos_matriz, axis=0)

    # 2. Recortar y centrar los ripples
    lista_ripples = []
    picos_centrados = []

    # Acomodar estructura según el formato de irips_python (2D o 1D)
    irips_array = np.asarray(irips_python)

    # CASO A: Forma de matriz 2D (parejas [inicio, fin])
    if irips_array.ndim == 2 and irips_array.shape[1] == 2:
        for inicio, fin in irips_array:
            segmento_orig = datos_filtrados[inicio:fin, :]

            if segmento_orig.shape[0] > 0:
                # Encontrar el canal con mayor amplitud máxima
                amplitud_por_canal = np.max(np.abs(segmento_orig), axis=0)
                canal_maximo = np.argmax(amplitud_por_canal)

                # Encontrar el pico máximo absoluto en la señal filtrada de ese canal
                pico_relativo = np.argmax(
                    np.abs(segmento_orig[:, canal_maximo])
                )
                idx_pico_global = inicio + pico_relativo

                nuevo_inicio = idx_pico_global - puntos_ventana
                nuevo_fin = idx_pico_global + puntos_ventana + 1

                if nuevo_inicio >= 0 and nuevo_fin <= puntos_de_tiempo:
                    segmento_centrado = datos_filtrados[
                        nuevo_inicio:nuevo_fin, :
                    ]
                    lista_ripples.append(segmento_centrado)
                    picos_centrados.append(idx_pico_global)

    # CASO B: Forma de vector 1D (índices centrales directos)
    else:
        for idx_pico_global in irips_array.flatten():
            nuevo_inicio = idx_pico_global - puntos_ventana
            nuevo_fin = idx_pico_global + puntos_ventana + 1

            if nuevo_inicio >= 0 and nuevo_fin <= puntos_de_tiempo:
                segmento_centrado = datos_filtrados[nuevo_inicio:nuevo_fin, :]
                lista_ripples.append(segmento_centrado)
                picos_centrados.append(idx_pico_global)

    if len(lista_ripples) == 0:
        print(
            f"⚠️ [{id_sesion}] Ningún ripple pudo centrarse correctamente dentro de los límites."
        )
        continue

    # Convertir a array de numpy
    matriz_ripples = np.array(lista_ripples)

    # CÁLCULO DEL RIPPLE MEDIO POR CANAL
    ripple_medio = np.mean(matriz_ripples, axis=0)

    # CÁLCULO DE LA AMPLITUD DE CADA RIPPLE INDIVIDUAL EN CADA CANAL
    amplitudes_por_ripple = np.ptp(matriz_ripples, axis=1)
    max_absoluta_por_canal = np.max(amplitudes_por_ripple, axis=0)
    promedio_indiv_por_canal = np.mean(amplitudes_por_ripple, axis=0)
    amplitud_ripple_medio = np.ptp(ripple_medio, axis=0)

    # Identificar el evento récord absoluto en la sesión
    idx_ripple_max, idx_canal_max = np.unravel_index(
        np.argmax(amplitudes_por_ripple), amplitudes_por_ripple.shape
    )
    canal_record = idx_canal_max + 1
    num_ripple_record = idx_ripple_max + 1
    valor_record = amplitudes_por_ripple[idx_ripple_max, idx_canal_max]

    # Guardar en DataFrame y CSV
    df_resultados = pd.DataFrame({
        "Canal": [f"Canal_{i+1}" for i in range(n_channels)],
        "Amplitud_Maxima_Absoluta": max_absoluta_por_canal,
        "Amplitud_Promedio_Ripples": promedio_indiv_por_canal,
        "Amplitud_Ripple_Medio": amplitud_ripple_medio,
    })

    ruta_csv = carpeta_resultados / "amplitud_maxima_por_canal.csv"
    df_resultados.to_csv(ruta_csv, index=False)

    # Guardar en memoria
    matrices_ripples_por_sesion[id_sesion] = matriz_ripples
    ripples_medios_por_sesion[id_sesion] = ripple_medio
    amplitudes_por_sesion[id_sesion] = amplitudes_por_ripple

    print(
        f"✓ [{id_sesion}] {len(matriz_ripples)} ripples centrados | Récord: Ch{canal_record} ({valor_record:.1f} uV) | CSV guardado"
    )

print(
    "\n¡Éxito! Métricas de amplitud calculadas y CSVs guardados para todas las sesiones."
)

Procesando y calculando métricas de ripples para 93 sesiones...

✓ [EFV_218_2026-01-12_18-02-06] 67 ripples centrados | Récord: Ch1 (2631.0 uV) | CSV guardado
✓ [EFV_218_2026-01-13_12-08-49] 25 ripples centrados | Récord: Ch13 (2689.5 uV) | CSV guardado
✓ [EFV_218_2026-01-13_13-09-16] 44 ripples centrados | Récord: Ch16 (2606.7 uV) | CSV guardado
✓ [EFV_218_2026-01-15_11-14-47] 30 ripples centrados | Récord: Ch10 (2734.6 uV) | CSV guardado
✓ [EFV_218_2026-01-15_11-56-40] 25 ripples centrados | Récord: Ch3 (2457.1 uV) | CSV guardado
✓ [EFV_218_2026-01-15_12-33-45] 66 ripples centrados | Récord: Ch16 (4401.1 uV) | CSV guardado
✓ [EFV_218_2026-01-16_17-50-35] 40 ripples centrados | Récord: Ch5 (2315.1 uV) | CSV guardado
✓ [EFV_220_2026-01-12_16-58-15] 105 ripples centrados | Récord: Ch9 (2592.6 uV) | CSV guardado
✓ [EFV_220_2026-01-13_14-12-19] 35 ripples centrados | Récord: Ch14 (2544.8 uV) | CSV guardado
✓ [EFV_220_2026-01-13_15-18-23] 29 ripples centrados | Récord: Ch8 (2382.9 uV) | CS

In [9]:
# FIGURAS PARA AMPLITUD Y FORMAS DE ONDA (MULTISESIÓN)

# Definir  el vector de tiempo en milisegundos (-30 a 30 ms)
# Usamos 'tiempo' de las celdas anteriores; si sus valores son menores a 1 (está en segundos), lo multiplicamos por 1000
if np.max(np.abs(tiempo)) <= 1:
    tiempo_ms = tiempo * 1000
else:
    tiempo_ms = tiempo

# Definir cuántas o cuáles sesiones quieres previsualizar en pantalla
# Opciones:
#   - Muestra de 3 sesiones: list(matrices_ripples_por_sesion.keys())[:3] Para que se vean todas solo hay que quitar el [:3]
#   - TODAS las sesiones:    list(matrices_ripples_por_sesion.keys())
sesiones_a_mostrar = list(matrices_ripples_por_sesion.keys())

print(
    f"Generando previsualización de gráficos para {len(sesiones_a_mostrar)} sesiones...\n"
)

for id_sesion in sesiones_a_mostrar:

    # Recuperar datos en memoria de la sesión
    matriz_ripples = matrices_ripples_por_sesion[id_sesion]
    ripple_medio = ripples_medios_por_sesion[id_sesion]
    amplitudes_por_ripple = amplitudes_por_sesion[id_sesion]
    carpeta_resultados = carpetas_resultados_por_sesion[id_sesion]

    # Recalcular las métricas necesarias para este gráfico
    max_absoluta_por_canal = np.max(amplitudes_por_ripple, axis=0)
    promedio_indiv_por_canal = np.mean(amplitudes_por_ripple, axis=0)
    canal_top_amp = np.argmax(promedio_indiv_por_canal) + 1

    # Crear la figura
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # --- SUBPLOT A: PERFIL DE AMPLITUD PICO A PICO POR CANAL ---
    canales_eje = np.arange(1, n_channels + 1)

    ax1.plot(
        promedio_indiv_por_canal,
        canales_eje,
        "o-",
        color="navy",
        linewidth=2,
        label="Amplitud Promedio",
    )
    ax1.plot(
        max_absoluta_por_canal,
        canales_eje,
        "s--",
        color="gray",
        alpha=0.5,
        label="Máximo Récord Indiv.",
    )

    ax1.axhline(
        y=canal_top_amp,
        color="crimson",
        linestyle="--",
        label=f"Máx Amplitud: Canal {canal_top_amp}",
    )

    ax1.invert_yaxis()
    ax1.set_title(
        f"[{id_sesion}]\nPerfil de Amplitud Pico a Pico por Canal",
        fontsize=11,
        fontweight="bold",
    )
    ax1.set_xlabel(r"Amplitud ($\mu V$)", fontsize=10)
    ax1.set_ylabel("Canal (Profundidad)", fontsize=10)
    ax1.grid(True, linestyle=":", alpha=0.6)
    ax1.legend(loc="lower right")

    # --- SUBPLOT B: FORMAS DE ONDA CORREGIDAS (TRACE PLOT) ---
    amp_oscilacion = np.max(np.abs(ripple_medio))
    if np.isnan(amp_oscilacion) or amp_oscilacion == 0:
        amp_oscilacion = 1.0  # Protección contra división por cero o NaNs

    offset = amp_oscilacion * 2.5

    posiciones_y = []
    etiquetas_canales = []

    for ch in range(n_channels):
        canal_num = ch + 1
        centro_canal = -ch * offset
        traza = ripple_medio[:, ch] + centro_canal

        # Plotear traza de la señal
        ax2.plot(tiempo_ms, traza, color="black", alpha=0.7, linewidth=1.2)

        posiciones_y.append(centro_canal)
        etiquetas_canales.append(f"Ch{canal_num}")

    # Configurar el eje Y para que muestre los nombres Ch1, Ch2... a la izquierda sin tocar las señales
    ax2.set_yticks(posiciones_y)
    ax2.set_yticklabels(etiquetas_canales, fontweight="bold", fontsize=9)

    # Destacar en rojo la etiqueta del canal con mayor amplitud
    for tick_label, ch_num in zip(ax2.get_yticklabels(), range(1, n_channels + 1)):
        if ch_num == canal_top_amp:
            tick_label.set_color("crimson")

    # Ajustar límites del eje X en milisegundos (-30 a 30)
    ax2.set_xlim([-ventana_ms, ventana_ms])

    ax2.axvline(
        x=0, color="gold", linestyle=":", alpha=0.8, label="Pico Ripple (0 ms)"
    )
    ax2.set_title(
        f"[{id_sesion}]\nFormas de Onda Filtradas (100-300 Hz) por Profundidad",
        fontsize=11,
        fontweight="bold",
    )
    ax2.set_xlabel("Tiempo respecto al pico (ms)", fontsize=10)
    ax2.set_ylabel("Canales Apilados (Superficie → Profundidad)", fontsize=10)
    ax2.grid(True, linestyle=":", alpha=0.3)
    ax2.legend(loc="upper right")

    plt.tight_layout()

    # --- GUARDAR FIGURA (DESCOMENTAR CUANDO QUIERAS GUARDAR EN DISCO) ---
    ruta_figura_amp = carpeta_resultados / "perfil_amplitud_y_formas_de_onda.png"
    plt.savefig(ruta_figura_amp, dpi=300)

    plt.close()

print(f"Gráficas de Amplitud y Formas de Onda guardada")

Generando previsualización de gráficos para 92 sesiones...

Gráficas de Amplitud y Formas de Onda guardada


In [10]:
# 7. FILTRADO EN BANDA RIPPLE Y CÁLCULO DEL RIPPLE POWER 

matrices_potencia_por_sesion = {}
power_medio_por_canal_por_sesion = {}

# Filtro Butterworth banda ripple (100-300 Hz)
f_low, f_high = 100, 300
sos = butter(4, [f_low, f_high], btype="band", fs=fs, output="sos")

print(
    f"Calculando Ripple Power MODO RÁPIDO para {len(sesiones_validas)} sesiones...\n"
)
for s in sesiones_validas:
    id_sesion = f"{s['grupo']}_{s['sesion']}"

    if id_sesion not in matrices_ripples_por_sesion:
        continue

    datos_matriz = datos_lfp_por_sesion[id_sesion]

    # Filtrar la matriz LFP completa (sosfiltfilt es súper rápido)
    lfp_ripple_band = sosfiltfilt(sos, datos_matriz, axis=0)

    irips_python = ripples_por_sesion[id_sesion]
    puntos_de_tiempo = datos_matriz.shape[0]

    lista_ripple_band_cortes = []

    # Recortar primero las ventanas filtradas
    irips_array = np.asarray(irips_python)

    if irips_array.ndim == 2 and irips_array.shape[1] == 2:
        for inicio, fin in irips_array:
            segmento_orig = lfp_ripple_band[inicio:fin, :]
            if segmento_orig.shape[0] > 0:
                amplitud_por_canal = np.max(np.abs(segmento_orig), axis=0)
                canal_maximo = np.argmax(amplitud_por_canal)
                pico_relativo = np.argmax(
                    np.abs(segmento_orig[:, canal_maximo])
                )
                idx_pico_global = inicio + pico_relativo

                nuevo_inicio = idx_pico_global - puntos_ventana
                nuevo_fin = idx_pico_global + puntos_ventana + 1

                if nuevo_inicio >= 0 and nuevo_fin <= puntos_de_tiempo:
                    lista_ripple_band_cortes.append(
                        lfp_ripple_band[nuevo_inicio:nuevo_fin, :]
                    )
    else:
        for idx_pico_global in irips_array.flatten():
            nuevo_inicio = idx_pico_global - puntos_ventana
            nuevo_fin = idx_pico_global + puntos_ventana + 1

            if nuevo_inicio >= 0 and nuevo_fin <= puntos_de_tiempo:
                lista_ripple_band_cortes.append(
                    lfp_ripple_band[nuevo_inicio:nuevo_fin, :]
                )

    if len(lista_ripple_band_cortes) == 0:
        continue

    # Convertir a matriz 3D: (n_ripples, tiempo, canales)
    matriz_ripple_band = np.array(lista_ripple_band_cortes)

    # HILBERT OPTIMIZADO: Calcular Hilbert solo sobre la matriz de ventana recortada en el eje temporal (axis=1)
    envolvente = np.abs(hilbert(matriz_ripple_band, axis=1))
    matriz_potencia = envolvente**2  # Potencia instantánea (uV^2)

    # Ripple Power medio por canal
    power_medio_por_canal = np.mean(matriz_potencia, axis=(0, 1))
    canal_max_power = np.argmax(power_medio_por_canal) + 1

    # Actualizar el archivo CSV en la carpeta ProcesamientoNatalia_antiguo de la sesión
    if ruta_csv.exists():
        df_resultados = pd.read_csv(ruta_csv)
        df_resultados["Ripple_Power_Medio"] = power_medio_por_canal
        df_resultados.to_csv(ruta_csv, index=False)
                 
    # Guardar variables en memoria
    matrices_potencia_por_sesion[id_sesion] = matriz_potencia
    power_medio_por_canal_por_sesion[id_sesion] = power_medio_por_canal
    
    print(
        f"✓ [{id_sesion}] Power calculado | Máx Power: Ch{canal_max_power} "
        f"({power_medio_por_canal[canal_max_power-1]:.2f} uV²)"
    )

print(
    "\n¡Éxito! Análisis de Ripple Power ultra-rápido completado para todas las sesiones."
)

Calculando Ripple Power MODO RÁPIDO para 93 sesiones...

✓ [EFV_218_2026-01-12_18-02-06] Power calculado | Máx Power: Ch9 (615907.84 uV²)
✓ [EFV_218_2026-01-13_12-08-49] Power calculado | Máx Power: Ch1 (858801.33 uV²)
✓ [EFV_218_2026-01-13_13-09-16] Power calculado | Máx Power: Ch4 (359043.71 uV²)
✓ [EFV_218_2026-01-15_11-14-47] Power calculado | Máx Power: Ch10 (670037.79 uV²)
✓ [EFV_218_2026-01-15_11-56-40] Power calculado | Máx Power: Ch10 (640130.38 uV²)
✓ [EFV_218_2026-01-15_12-33-45] Power calculado | Máx Power: Ch8 (785482.46 uV²)
✓ [EFV_218_2026-01-16_17-50-35] Power calculado | Máx Power: Ch9 (533135.44 uV²)
✓ [EFV_220_2026-01-12_16-58-15] Power calculado | Máx Power: Ch2 (259232.91 uV²)
✓ [EFV_220_2026-01-13_14-12-19] Power calculado | Máx Power: Ch2 (571947.62 uV²)
✓ [EFV_220_2026-01-13_15-18-23] Power calculado | Máx Power: Ch16 (475385.26 uV²)
✓ [EFV_220_2026-01-14_11-44-04] Power calculado | Máx Power: Ch12 (603201.05 uV²)
✓ [EFV_220_2026-01-14_12-37-30] Power calculado 

In [11]:
# FIGURA MULTIPANEL: ESPECTROGRAMA + PERFIL DE POTENCIA 

# Previsualizar solo las últimas 3 sesiones guardadas en memoria
sesiones_a_mostrar = list(matrices_potencia_por_sesion.keys())
print(
    f"Generando figuras de Espectrograma + Power para las últimas {len(sesiones_a_mostrar)} sesiones:\n"
    f"{sesiones_a_mostrar}\n"
)

for id_sesion in sesiones_a_mostrar:

    # Recuperar datos necesarios de la sesión desde el almacenamiento en memoria
    ripple_medio = ripples_medios_por_sesion[id_sesion]
    power_medio_por_canal = power_medio_por_canal_por_sesion[id_sesion]
    carpeta_resultados = carpetas_resultados_por_sesion[id_sesion]

    # Recomputar canal con mayor potencia promedio
    canal_max_power = np.argmax(power_medio_por_canal) + 1

    # Crear figura multipanel
    plt.figure(figsize=(14, 5))

    # --- SUBPLOT A: PERFIL DE RIPPLE POWER MEDIO POR CANAL ---
    plt.subplot(1, 2, 1)
    canales_eje = np.arange(1, n_channels + 1)

    plt.plot(
        power_medio_por_canal,
        canales_eje,
        "o-",
        color="crimson",
        linewidth=2,
        markersize=7,
        label="Power Medio",
    )
    plt.axhline(
        y=canal_max_power,
        color="gold",
        linestyle="--",
        label=f"Máx Power: Canal {canal_max_power}",
    )

    plt.gca().invert_yaxis()  # Invertimos eje Y para simular profundidad
    plt.title(
        f"[{id_sesion}]\nPerfil de Ripple Power Medio (100-300 Hz) por Canal",
        fontsize=11,
        fontweight="bold",
    )
    plt.xlabel(r"Potencia Media ($\mu V^2$)", fontsize=10)
    plt.ylabel("Canal (Profundidad)", fontsize=10)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="lower right")

    # --- SUBPLOT B: ESPECTROGRAMA TIEMPO-FRECUENCIA DEL CANAL MÁXIMO ---
    plt.subplot(1, 2, 2)

    # Extraemos la señal del canal de mayor potencia
    señal_espectrograma = ripple_medio[:, canal_max_power - 1]

    frecuencias, tiempos_spec, Sxx = spectrogram(
        señal_espectrograma,
        fs=fs,
        nperseg=64,
        noverlap=60,
        nfft=256,
    )

    # Convertimos eje de tiempo a milisegundos centrados en 0 (-30 ms a +30 ms)
    tiempos_spec_ms = (tiempos_spec * 1000) - ventana_ms

    plt.pcolormesh(
        tiempos_spec_ms, frecuencias, Sxx, shading="gouraud", cmap="jet"
    )
    plt.axhline(y=100, color="white", linestyle=":", alpha=0.7)
    plt.axhline(
        y=300,
        color="white",
        linestyle=":",
        alpha=0.7,
        label="Banda Ripple (100-300 Hz)",
    )
    plt.ylim(0, 400)  # Mostramos de 0 a 400 Hz
    plt.title(
        f"[{id_sesion}]\nEspectrograma Tiempo-Frecuencia (Canal {canal_max_power})",
        fontsize=11,
        fontweight="bold",
    )
    plt.xlabel("Tiempo respecto al pico (ms)", fontsize=10)
    plt.ylabel("Frecuencia (Hz)", fontsize=10)
    plt.colorbar(label="Densidad Espectral de Potencia")
    plt.legend(loc="upper right")

    plt.tight_layout()

    # --- GUARDAR FIGURA (DESCOMENTAR PARA GUARDAR EN TODAS LAS CARPETAS) ---
    ruta_figura_power = carpeta_resultados / "ripple_power_y_espectrograma.png"
    plt.savefig(ruta_figura_power, dpi=300)

    # Mostrar figura en la pantalla
    plt.close()

print("Previsualización de Espectrograma + Power finalizada.")

Generando figuras de Espectrograma + Power para las últimas 92 sesiones:
['EFV_218_2026-01-12_18-02-06', 'EFV_218_2026-01-13_12-08-49', 'EFV_218_2026-01-13_13-09-16', 'EFV_218_2026-01-15_11-14-47', 'EFV_218_2026-01-15_11-56-40', 'EFV_218_2026-01-15_12-33-45', 'EFV_218_2026-01-16_17-50-35', 'EFV_220_2026-01-12_16-58-15', 'EFV_220_2026-01-13_14-12-19', 'EFV_220_2026-01-13_15-18-23', 'EFV_220_2026-01-14_11-44-04', 'EFV_220_2026-01-14_12-37-30', 'EFV_220_2026-01-15_17-57-21', 'EFV_220_2026-01-15_18-37-42', 'EFV_220_2026-01-16_16-53-48', 'EFV_263_2026-02-10_15-35-14', 'EFV_263_2026-02-10_16-14-52', 'EFV_277_2026-02-16_17-04-40', 'EFV_277_2026-02-17_14-36-39', 'EFV_277_2026-02-17_15-19-53', 'EFV_277_2026-02-18_14-28-50', 'EFV_277_2026-02-19_17-00-23', 'EFV_277_2026-02-19_17-17-38', 'EFV_277_2026-02-20_12-02-47', 'EFV_279_2026-02-16_15-35-30', 'EFV_279_2026-02-17_12-53-11', 'EFV_279_2026-02-17_13-36-18', 'EFV_279_2026-02-18_12-40-41', 'EFV_279_2026-02-20_10-36-28', 'EFV_279_2026-02-20_11-07-3

In [12]:
# 8. CÁLCULO DE CSD MEDIO POR CANAL EN EL RIPPLE Y PERFIL VERTICAL (MULTISESIÓN)

# Diccionarios para almacenar en memoria los datos CSD procesados
csd_matrices_por_sesion = {}
csd_medios_por_sesion = {}
tiempos_csd_por_sesion = {}

print(
    f"Procesando CSD para {len(sesiones_validas)} sesiones válidas...\n"
)

for s in sesiones_validas:
    id_sesion = f"{s['grupo']}_{s['sesion']}"
    archivo_csd = s.get("csd") or s.get("csd_analysis")
    carpeta_resultados = carpetas_resultados_por_sesion[id_sesion]

    if not archivo_csd or not archivo_csd.exists():
        print(f"⚠️ [{id_sesion}] Archivo CSD no encontrado. Se omite.")
        continue

    try:
        # Cargar el archivo .mat usando scipy.io
        mat_csd = sio.loadmat(archivo_csd)

        # Acceder a la estructura 'rippleCSDs'
        ripple_struct = mat_csd["rippleCSDs"][0, 0]

        # Extraer vector de tiempo y matriz 3D (tiempo, canales, nºripples)
        tiempo_csd = ripple_struct["t"].flatten()
        csd_raw = ripple_struct["CSDs"]

        # Promediar sobre ripples (eje 2) y transponer (.T) -> [canales, tiempo]
        csd_matrix = -1 * np.mean(csd_raw, axis=2).T

        # Extraer dimensiones dinámicamente
        n_canales_csd, n_muestras = csd_matrix.shape

        # Definir la ventana central del ripple
        centro = n_muestras // 2
        ancho_ventana = int(n_muestras * 0.2)
        csd_ventana = csd_matrix[
            :, centro - ancho_ventana : centro + ancho_ventana
        ]

        # Calcular promedio por canal para el perfil vertical
        csd_medio_canal = np.mean(csd_ventana, axis=1)

        # Guardar objetos en memoria
        csd_matrices_por_sesion[id_sesion] = csd_matrix
        csd_medios_por_sesion[id_sesion] = csd_medio_canal
        tiempos_csd_por_sesion[id_sesion] = tiempo_csd

        print(
            f"✓ [{id_sesion}] CSD procesado con éxito ({n_canales_csd} canales, {csd_raw.shape[2]} ripples)"
        )

    except Exception as e:
        print(f"❌ [{id_sesion}] Error al procesar CSD: {e}")

print(
    f"\n¡Éxito! CSD cargado en memoria para {len(csd_matrices_por_sesion)} sesiones."
)

Procesando CSD para 93 sesiones válidas...

✓ [EFV_218_2026-01-12_18-02-06] CSD procesado con éxito (14 canales, 67 ripples)
✓ [EFV_218_2026-01-13_12-08-49] CSD procesado con éxito (14 canales, 25 ripples)
✓ [EFV_218_2026-01-13_13-09-16] CSD procesado con éxito (14 canales, 44 ripples)
✓ [EFV_218_2026-01-15_11-14-47] CSD procesado con éxito (14 canales, 30 ripples)
✓ [EFV_218_2026-01-15_11-56-40] CSD procesado con éxito (14 canales, 25 ripples)
✓ [EFV_218_2026-01-15_12-33-45] CSD procesado con éxito (14 canales, 66 ripples)
✓ [EFV_218_2026-01-16_17-50-35] CSD procesado con éxito (14 canales, 40 ripples)
✓ [EFV_220_2026-01-12_16-58-15] CSD procesado con éxito (14 canales, 105 ripples)
✓ [EFV_220_2026-01-13_14-12-19] CSD procesado con éxito (14 canales, 35 ripples)
✓ [EFV_220_2026-01-13_15-18-23] CSD procesado con éxito (14 canales, 29 ripples)
✓ [EFV_220_2026-01-14_11-44-04] CSD procesado con éxito (14 canales, 104 ripples)
✓ [EFV_220_2026-01-14_12-37-30] CSD procesado con éxito (14 can

c:\Users\natal\anaconda3\envs\niemannpick\Lib\site-packages\numpy\_core\fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\natal\anaconda3\envs\niemannpick\Lib\site-packages\numpy\_core\_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


✓ [NPC_250_2026-02-05_17-17-46] CSD procesado con éxito (14 canales, 28 ripples)
✓ [NPC_262_2026-02-10_11-40-21] CSD procesado con éxito (14 canales, 164 ripples)
✓ [NPC_262_2026-02-10_12-33-48] CSD procesado con éxito (14 canales, 94 ripples)
✓ [NPC_262_2026-02-10_12-55-22] CSD procesado con éxito (14 canales, 91 ripples)
✓ [NPC_262_2026-02-11_11-04-49] CSD procesado con éxito (14 canales, 43 ripples)
✓ [NPC_262_2026-02-11_12-05-14] CSD procesado con éxito (14 canales, 11 ripples)
✓ [NPC_262_2026-02-11_12-46-19] CSD procesado con éxito (14 canales, 74 ripples)
✓ [NPC_262_2026-02-11_15-41-04] CSD procesado con éxito (14 canales, 64 ripples)
✓ [NPC_262_2026-02-11_16-24-13] CSD procesado con éxito (14 canales, 95 ripples)
✓ [NPC_262_2026-02-12_16-44-33] CSD procesado con éxito (14 canales, 33 ripples)
✓ [NPC_262_2026-02-12_17-06-58] CSD procesado con éxito (14 canales, 151 ripples)
✓ [NPC_262_2026-02-13_12-03-07] CSD procesado con éxito (14 canales, 30 ripples)
✓ [NPC_262_2026-02-13_12-3

In [13]:
# VISUALIZACIÓN DE CSD MEDIO Y MAPA ESPACIO-TEMPORAL 

sesiones_a_mostrar = list(csd_matrices_por_sesion.keys())

print(
    f"Generando gráficos CSD para las últimas {len(sesiones_a_mostrar)} sesiones:\n"
    f"{sesiones_a_mostrar}\n"
)

for id_sesion in sesiones_a_mostrar:

    # Recuperar datos en memoria de la sesión
    csd_matrix = csd_matrices_por_sesion[id_sesion]
    csd_medio_canal = csd_medios_por_sesion[id_sesion]
    tiempo = tiempos_csd_por_sesion[id_sesion]
    carpeta_resultados = carpetas_resultados_por_sesion[id_sesion]

    n_canales_csd = csd_matrix.shape[0]
    canales = np.arange(1, n_canales_csd + 1)

    # Representación Gráfica
    plt.figure(figsize=(12, 5))

    # --- Subplot 1: Perfil de Fuentes y Sumideros ---
    plt.subplot(1, 2, 1)
    plt.plot(
        csd_medio_canal,
        canales,
        "-o",
        color="black",
        linewidth=1.5,
        markersize=4,
    )
    plt.axvline(0, color="red", linestyle="--", alpha=0.7, label="Línea cero")
    plt.gca().invert_yaxis()  # Canal 1 en la parte superior (superficie)
    plt.xlabel("CSD Promedio (Fuentes > 0 | Sumideros < 0)")
    plt.ylabel("Canal / Profundidad")
    plt.title(f"[{id_sesion}]\nPerfil CSD Medio ({n_canales_csd} Canales)")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend()

    # --- Subplot 2: Mapa Espacio-Temporal 2D ---
    plt.subplot(1, 2, 2)
    vmax = np.max(np.abs(csd_matrix))  # Escala simétrica para centrar el cero

    im = plt.imshow(
        csd_matrix,
        aspect="auto",
        cmap="seismic",
        vmin=-vmax,
        vmax=vmax,
        extent=[tiempo[0], tiempo[-1], n_canales_csd, 1],
    )

    cbar = plt.colorbar(im)
    cbar.set_label("CSD")
    plt.axvline(
        0, color="black", linestyle="--", alpha=0.7, label="Pico del Ripple"
    )
    plt.xlabel("Tiempo (muestras respecto al pico)")
    plt.ylabel("Canal")
    plt.title(f"[{id_sesion}]\nMapa Espacio-Temporal CSD")
    plt.legend(loc="upper right")

    plt.tight_layout()

    # --- GUARDAR FIGURA (DESCOMENTAR PARA GUARDAR EN TODAS LAS CARPETAS) ---
    ruta_figura_csd = carpeta_resultados / "perfil_CSD_medio_y_mapa_espacio_temporal_CSD.png"
    plt.savefig(ruta_figura_csd, dpi=300)

    plt.close()

print("Previsualización de CSD finalizada.")

Generando gráficos CSD para las últimas 93 sesiones:
['EFV_218_2026-01-12_18-02-06', 'EFV_218_2026-01-13_12-08-49', 'EFV_218_2026-01-13_13-09-16', 'EFV_218_2026-01-15_11-14-47', 'EFV_218_2026-01-15_11-56-40', 'EFV_218_2026-01-15_12-33-45', 'EFV_218_2026-01-16_17-50-35', 'EFV_220_2026-01-12_16-58-15', 'EFV_220_2026-01-13_14-12-19', 'EFV_220_2026-01-13_15-18-23', 'EFV_220_2026-01-14_11-44-04', 'EFV_220_2026-01-14_12-37-30', 'EFV_220_2026-01-15_17-57-21', 'EFV_220_2026-01-15_18-37-42', 'EFV_220_2026-01-16_16-53-48', 'EFV_263_2026-02-10_15-35-14', 'EFV_263_2026-02-10_16-14-52', 'EFV_277_2026-02-16_17-04-40', 'EFV_277_2026-02-17_14-36-39', 'EFV_277_2026-02-17_15-19-53', 'EFV_277_2026-02-18_14-28-50', 'EFV_277_2026-02-19_17-00-23', 'EFV_277_2026-02-19_17-17-38', 'EFV_277_2026-02-20_12-02-47', 'EFV_279_2026-02-16_15-35-30', 'EFV_279_2026-02-17_12-53-11', 'EFV_279_2026-02-17_13-36-18', 'EFV_279_2026-02-18_12-40-41', 'EFV_279_2026-02-20_10-36-28', 'EFV_279_2026-02-20_11-07-31', 'EFV_280_2026-02

In [14]:
# ==============================================================================
# CELDA FINAL: FIGURA INTEGRADA Y DETECCIÓN ANATÓMICA DE CAPAS
# ==============================================================================

# ALGORITMO PONDERADO DE DETECCIÓN ANATÓMICA (Pyr, Rad, SLM, Or)
def clasificar_capas_hipocampo_ponderado(csd_medio, power_medio, canales_csd):
    n_csd = len(csd_medio)
    ch_max_power = np.argmax(power_medio) + 1

    indices_fuentes = [
        i
        for i in range(1, n_csd - 1)
        if csd_medio[i] > csd_medio[i - 1]
        and csd_medio[i] > csd_medio[i + 1]
        and csd_medio[i] > 0
    ]

    if not indices_fuentes:
        idx_pyr_csd = np.argmax(csd_medio)
    else:
        puntuaciones = []
        for idx in indices_fuentes:
            ch_real = canales_csd[idx]
            amplitud_fuente = csd_medio[idx]
            distancia = abs(ch_real - ch_max_power)
            score = amplitud_fuente / (1.0 + distancia)
            puntuaciones.append(score)

        idx_pyr_csd = indices_fuentes[np.argmax(puntuaciones)]

    ch_pyr = canales_csd[idx_pyr_csd]

    idx_rad_csd = next(
        (
            i
            for i in range(idx_pyr_csd + 1, n_csd - 1)
            if csd_medio[i] < csd_medio[i - 1]
            and csd_medio[i] < csd_medio[i + 1]
        ),
        None,
    )
    ch_rad = canales_csd[idx_rad_csd] if idx_rad_csd is not None else None

    idx_slm_csd = None
    if idx_rad_csd is not None:
        idx_slm_csd = next(
            (
                i
                for i in range(idx_rad_csd + 1, n_csd - 1)
                if csd_medio[i] > csd_medio[i - 1]
                and csd_medio[i] > csd_medio[i + 1]
            ),
            None,
        )
    ch_slm = canales_csd[idx_slm_csd] if idx_slm_csd is not None else None

    idx_or_csd = next(
        (
            i
            for i in range(idx_pyr_csd - 1, 0, -1)
            if csd_medio[i] < csd_medio[i + 1]
            and csd_medio[i] < csd_medio[i - 1]
        ),
        None,
    )
    ch_or = canales_csd[idx_or_csd] if idx_or_csd is not None else None

    return ch_pyr, ch_rad, ch_slm, ch_or


# SELECCIÓN DE LAS SESIONES 
sesiones_procesadas = [
    id_s
    for id_s in csd_matrices_por_sesion.keys()
    if id_s in ripples_medios_por_sesion
    and id_s in power_medio_por_canal_por_sesion
    and id_s in amplitudes_por_sesion
]

print(
    f"Generando figura integrada para {len(sesiones_procesadas)} sesiones completas...\n"
)

for id_sesion in sesiones_procesadas:

    # Recuperar datos en memoria de la sesión
    ripple_medio_filtrado = ripples_medios_por_sesion[id_sesion]
    csd_matrix = csd_matrices_por_sesion[id_sesion]
    csd_medio_canal = csd_medios_por_sesion[id_sesion]
    tiempo = tiempos_csd_por_sesion[id_sesion]
    promedio_indiv_por_canal = amplitudes_por_sesion[id_sesion].mean(axis=0)
    max_absoluta_por_canal = amplitudes_por_sesion[id_sesion].max(axis=0)
    power_medio_por_canal = power_medio_por_canal_por_sesion[id_sesion]
    carpeta_resultados = carpetas_resultados_por_sesion[id_sesion]

    # GESTIÓN Y AJUSTE DE DIMENSIONES (TIEMPO Y CANALES)
    n_samples_lfp, n_ch_lfp = ripple_medio_filtrado.shape
    canales_lfp = np.arange(1, n_ch_lfp + 1)

    mediana_amp = np.median(promedio_indiv_por_canal)
    umbral_ruido = mediana_amp * 5

    canales_limpios_mask = promedio_indiv_por_canal < umbral_ruido
    if not np.any(canales_limpios_mask):
        canales_limpios_mask = np.ones_like(promedio_indiv_por_canal, dtype=bool)

    power_filtrado_ruido = power_medio_por_canal.copy()
    power_filtrado_ruido[~canales_limpios_mask] = 0
    canal_max_power = np.argmax(power_filtrado_ruido) + 1

    # Ajuste dinámico del vector tiempo
    if len(tiempo) != n_samples_lfp:
        tiempo = np.linspace(tiempo[0], tiempo[-1], n_samples_lfp)

    n_ch_csd = csd_matrix.shape[0]

    # Centrar canales de CSD respecto a los de LFP
    ch_offset_csd = (n_ch_lfp - n_ch_csd) // 2
    canales_csd = np.arange(1 + ch_offset_csd, n_ch_csd + 1 + ch_offset_csd)

    n_ch_amp = len(promedio_indiv_por_canal)
    canales_amp = np.arange(1, n_ch_amp + 1)

    n_ch_power = len(power_medio_por_canal)
    canales_power = np.arange(1, n_ch_power + 1)

    canal_top_amp = np.argmax(promedio_indiv_por_canal) + 1
    canal_max_power = np.argmax(power_medio_por_canal) + 1

    # CLASIFICACIÓN DE CAPAS
    ch_pyr, ch_rad, ch_slm, ch_or = clasificar_capas_hipocampo_ponderado(
        csd_medio_canal, power_medio_por_canal, canales_csd
    )

    capas_dibujo = [
        (ch_pyr, "Pyr", "#ffe0b2"),
        (ch_rad, "Rad", "#ffcdd2"),
        (ch_slm, "SLM", "#e1bee7"),
        (ch_or, "Or", "#bbdefb"),
    ]
    capas_dibujo = [
        (ch, lbl, col) for ch, lbl, col in capas_dibujo if ch is not None
    ]

    # ==============================
    # GUARDAR ANATOMÍA EN CSV 
    # ==============================
    ruta_csv = carpeta_resultados / "amplitud_maxima_por_canal.csv"
    if ruta_csv.exists():
        df_resultados = pd.read_csv(ruta_csv)
        capa_por_canal = [""] * len(df_resultados)
        mapping_capas = {ch_pyr: "Pyr", ch_rad: "Rad", ch_slm: "SLM", ch_or: "Or"}
        for ch_idx, row in df_resultados.iterrows():
            ch_num = ch_idx + 1
            if ch_num in mapping_capas:
                capa_por_canal[ch_idx] = mapping_capas[ch_num]
        df_resultados["Capa_Hipocampo"] = capa_por_canal
        df_resultados.to_csv(ruta_csv, index=False)

    # GENERACIÓN DE LA FIGURA MULTI-PANEL
    fig, axes = plt.subplots(
        1,
        4,
        figsize=(18, 9),
        sharey=True,
        gridspec_kw={"width_ratios": [3.5, 1, 1, 1], "wspace": 0.12},
    )

    ax_main, ax_csd_prof, ax_amp, ax_power = axes

    # --- SUBPLOT 1: MAPA CSD 2D + LFP FILTRADO (100-300 Hz) ---
    vmax = np.max(np.abs(csd_matrix))

    y_top = canales_csd[0] - 0.5
    y_bottom = canales_csd[-1] + 0.5

    im = ax_main.imshow(
        csd_matrix,
        aspect="auto",
        cmap="seismic",
        vmin=-vmax,
        vmax=vmax,
        extent=[tiempo[0], tiempo[-1], y_bottom, y_top],
    )

    # Sombras anatómicas y etiquetas
    for ch_num, label, color in capas_dibujo:
        ax_main.axhspan(
            ch_num - 0.5, ch_num + 0.5, color=color, alpha=0.35, zorder=1
        )
        ax_main.text(
            tiempo[0] - (tiempo[-1] - tiempo[0]) * 0.12,
            ch_num,
            label,
            color="black",
            fontsize=9,
            fontweight="bold",
            va="center",
            ha="right",
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                edgecolor="gray",
                alpha=0.9,
            ),
            clip_on=False,
        )

    # Trazo LFP Filtrado
    amp_oscilacion = np.max(np.abs(ripple_medio_filtrado))
    escala_lfp = 0.8 / amp_oscilacion if amp_oscilacion > 0 else 1.0

    for ch in range(n_ch_lfp):
        canal_num = ch + 1
        traza_escalada = canal_num - (
            ripple_medio_filtrado[:, ch] * escala_lfp
        )
        ax_main.plot(
            tiempo,
            traza_escalada,
            color="black",
            alpha=0.85,
            linewidth=1.0,
            zorder=3,
        )

    ax_main.axvline(
        0,
        color="gold",
        linestyle=":",
        linewidth=1.5,
        label="Pico Ripple (0 ms)",
        zorder=4,
    )
    ax_main.set_title(
        f"[{id_sesion}]\nCSD Espacio-Temporal + LFP Filtrado (100-300 Hz)",
        fontsize=11,
        fontweight="bold",
    )
    ax_main.set_xlabel("Tiempo respecto al pico (ms)", fontsize=10)

    ax_main.set_ylabel(
        "Canal (Profundidad)", fontsize=10, fontweight="bold", labelpad=80
    )

    cbar = fig.colorbar(im, ax=ax_main, orientation="vertical", pad=0.015)
    cbar.set_label("CSD", fontsize=9)

    # --- SUBPLOT 2: PERFIL CSD MEDIO ---
    f_interp = interp1d(canales_csd, csd_medio_canal, kind="cubic")
    canales_finos = np.linspace(canales_csd[0], canales_csd[-1], 300)
    csd_fino = f_interp(canales_finos)

    ax_csd_prof.plot(csd_fino, canales_finos, color="black", linewidth=1.5)
    ax_csd_prof.scatter(
        csd_medio_canal, canales_csd, color="black", s=20, zorder=5
    )
    ax_csd_prof.axvline(0, color="red", linestyle="--", alpha=0.7)

    ax_csd_prof.fill_betweenx(
        canales_finos,
        csd_fino,
        0,
        where=(csd_fino > 0),
        color="red",
        alpha=0.35,
        label="Fuente (>0)",
    )
    ax_csd_prof.fill_betweenx(
        canales_finos,
        csd_fino,
        0,
        where=(csd_fino < 0),
        color="blue",
        alpha=0.35,
        label="Sumidero (<0)",
    )

    ax_csd_prof.set_title("Perfil CSD Medio", fontsize=11, fontweight="bold")
    ax_csd_prof.set_xlabel("CSD Promedio", fontsize=9)
    ax_csd_prof.grid(True, linestyle=":", alpha=0.6)
    ax_csd_prof.legend(loc="lower right", fontsize=7)

    # CONTROL DE CANALES RUIDOSOS
    # DETECCIÓN AUTOMÁTICA DE "BAD CHANNELS" (Artefactos masivos)
    mediana_amp = np.median(promedio_indiv_por_canal)
    umbral_ruido = mediana_amp * 5  # Canales con > 5x la mediana son artefactos

    # Máscara booleana con los canales limpios/válidos
    canales_limpios_mask = promedio_indiv_por_canal < umbral_ruido

    # Si todos salieran ruidosos por algún motivo, mantenemos todos para evitar error
    if not np.any(canales_limpios_mask):
        canales_limpios_mask = np.ones_like(promedio_indiv_por_canal, dtype=bool)

    # Recalcular el Canal Máximo de Power IGNORANDO el ruido de artefactos
    power_filtrado_ruido = power_medio_por_canal.copy()
    power_filtrado_ruido[~canales_limpios_mask] = 0
    canal_max_power = np.argmax(power_filtrado_ruido) + 1

    # --- SUBPLOT 3: PERFIL DE AMPLITUD PEAK-TO-PEAK ---
    ax_amp.plot(
        promedio_indiv_por_canal,
        canales_amp,
        "o-",
        color="navy",
        linewidth=1.8,
        markersize=4,
        label="Promedio",
    )
    ax_amp.plot(
        max_absoluta_por_canal,
        canales_amp,
        "s--",
        color="gray",
        alpha=0.6,
        markersize=3,
        label="Máxima",
    )
    ax_amp.axhline(
        y=canal_top_amp,
        color="crimson",
        linestyle="--",
        alpha=0.8,
        label=f"Máx: Ch{canal_top_amp}",
    )

    # Ajuste riguroso de escala: basado en la máxima amplitud biológica VÁLIDA
    max_amp_valida = np.max(promedio_indiv_por_canal[canales_limpios_mask])
    ax_amp.set_xlim(0, max_amp_valida * 1.35)

    ax_amp.set_title("Amplitud Peak-to-Peak", fontsize=11, fontweight="bold")
    ax_amp.set_xlabel(r"Amplitud ($\mu$V)", fontsize=9)
    ax_amp.grid(True, linestyle=":", alpha=0.6)
    ax_amp.legend(loc="lower right", fontsize=7)


    # --- SUBPLOT 4: PERFIL DE RIPPLE POWER MEDIO ---
    ax_power.plot(
        power_medio_por_canal,
        canales_power,
        "o-",
        color="crimson",
        linewidth=1.8,
        markersize=4,
    )
    ax_power.axhline(
        y=canal_max_power,
        color="gold",
        linestyle="--",
        alpha=0.9,
        label=f"Máx Power: Ch{canal_max_power}",
    )

    # Ajuste riguroso de escala para potencia
    max_power_valido = np.max(power_medio_por_canal[canales_limpios_mask])
    ax_power.set_xlim(0, max_power_valido * 1.35)

    ax_power.set_title("Ripple Power Medio", fontsize=11, fontweight="bold")
    ax_power.set_xlabel(r"Potencia ($\mu$V$^2$)", fontsize=9)
    ax_power.grid(True, linestyle=":", alpha=0.6)
    ax_power.legend(loc="lower right", fontsize=7)

    # CONFIGURACIÓN DEL EJE Y
    ax_main.set_ylim(n_ch_lfp + 0.5, 0.5)

    labels_eje_y = [f"Ch{ch}" for ch in canales_lfp]

    ax_main.set_yticks(canales_lfp)
    ax_main.set_yticklabels(labels_eje_y, fontsize=8, fontweight="bold")

    ax_csd_prof.set_yticks(canales_lfp)
    ax_csd_prof.set_yticklabels(labels_eje_y, fontsize=8, fontweight="bold")
    ax_csd_prof.tick_params(labelleft=True)

    for ax in [ax_amp, ax_power]:
        ax.tick_params(labelleft=False)

    plt.subplots_adjust(wspace=0.35, hspace=0.3)
    plt.tight_layout()

        # --- GUARDAR FIGURA (DESCOMENTAR PARA GUARDAR EN DISCO) ---
    ruta_figura_final = carpeta_resultados / "perfil_integrado_ripples_CSD.png"
    plt.savefig(ruta_figura_final, dpi=300, bbox_inches='tight')

    # --- ALINEAR CSD CON LFP (Rellenar canales perdidos por bordes) ---
    csd_completo = np.full(n_ch_lfp, np.nan)
    if csd_medio_canal is not None:
        n_ch_csd = len(csd_medio_canal)
        ch_offset = (n_ch_lfp - n_ch_csd) // 2
        csd_completo[ch_offset : ch_offset + n_ch_csd] = csd_medio_canal

    # --- CREAR Y GUARDAR EL CSV INDIVIDUAL CON QC ---
    df_canal = pd.DataFrame(
        {
            "Canal": np.arange(1, n_ch_lfp + 1),
            "Amplitud_Media_uV": promedio_indiv_por_canal,
            "Amplitud_Max_uV": max_absoluta_por_canal,
            "Ripple_Power_uV2": power_medio_por_canal,
            "CSD_Medio": csd_completo,
            "QC_Status": [
                "OK" if limpio else "BAD_CHANNEL"
                for limpio in canales_limpios_mask
            ],
        }
    )

    ruta_csv_individual = carpeta_resultados / "amplitud_maxima_por_canal.csv"
    df_canal.to_csv(ruta_csv_individual, index=False)

    plt.close()

print("Previsualización de figura integrada finalizada.")

Generando figura integrada para 92 sesiones completas...



C:\Users\natal\AppData\Local\Temp\ipykernel_22280\3352849389.py:380: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\natal\AppData\Local\Temp\ipykernel_22280\3352849389.py:380: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\natal\AppData\Local\Temp\ipykernel_22280\3352849389.py:380: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\natal\AppData\Local\Temp\ipykernel_22280\3352849389.py:380: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\natal\AppData\Local\Temp\ipykernel_22280\3352849389.py:380: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.

Previsualización de figura integrada finalizada.


In [15]:
# GUARDAR DATOS EN EXCEL CON RESUMEN GLOBAL DE TODAS LAS SESIONES
# Definir la carpeta principal en la ruta del disco D:
carpeta_principal = Path(r"D:\Violeta\ANALISIS")
carpeta_salida_global = carpeta_principal / "ProcesamientoNatalia_antiguo"
carpeta_salida_global.mkdir(parents=True, exist_ok=True)

registros_globales = []

print(
    f"Compilando resumen global de todas las sesiones válidas en: {carpeta_salida_global}...\n"
)

for s in sesiones_validas:
    id_sesion = f"{s['grupo']}_{s['sesion']}"

    if id_sesion not in matrices_ripples_por_sesion:
        continue

    # Recuperar datos procesados previamente
    grupo = s["grupo"]
    raton = s.get("raton", s["grupo"])
    sesion = s["sesion"]

    # Matrices y vectores de la sesión
    ripples = ripples_por_sesion[id_sesion]
    num_ripples = len(ripples)

    csd_medio = csd_medios_por_sesion.get(id_sesion)
    power_medio = power_medio_por_canal_por_sesion.get(id_sesion)
    amp_medio = amplitudes_por_sesion[id_sesion].mean(axis=0)

    n_ch_lfp = len(amp_medio)
    n_ch_csd = len(csd_medio) if csd_medio is not None else n_ch_lfp
    ch_offset_csd = (n_ch_lfp - n_ch_csd) // 2
    canales_csd = np.arange(1 + ch_offset_csd, n_ch_csd + 1 + ch_offset_csd)

    # CONTROL DE CANALES RUIDOSOS
    mediana_amp = np.median(amp_medio)
    canales_limpios_mask = amp_medio < (mediana_amp * 5)

    if not np.any(canales_limpios_mask):
        canales_limpios_mask = np.ones_like(amp_medio, dtype=bool)

    power_filtrado_ruido = power_medio.copy()
    power_filtrado_ruido[~canales_limpios_mask] = 0

    # Determinar el canal con máximo de potencia ignorando el ruido
    canal_max_power = np.argmax(power_filtrado_ruido) + 1
    max_power_val = power_medio[canal_max_power - 1]

    # Detección de capas
    ch_pyr, ch_rad, ch_slm, ch_or = clasificar_capas_hipocampo_ponderado(
        csd_medio, power_medio, canales_csd
    )

    # Extraer métricas en Capa Piramidal
    pyr_power = power_medio[ch_pyr - 1] if ch_pyr is not None else np.nan
    pyr_amp = amp_medio[ch_pyr - 1] if ch_pyr is not None else np.nan

    # Canales con máximos globales
    canal_max_power = np.argmax(power_medio) + 1
    max_power_val = power_medio[canal_max_power - 1]

    # Crear fila para la sesión
    fila = {
        "Grupo": grupo,
        "Raton": raton,
        "Sesion": sesion,
        "Num_Ripples": num_ripples,
        "Canal_Pyr": ch_pyr,
        "Canal_Rad": ch_rad,
        "Canal_SLM": ch_slm,
        "Canal_Or": ch_or,
        "Pyr_Ripple_Power": pyr_power,
        "Pyr_Amp_Mean": pyr_amp,
        "Max_Ripple_Power": max_power_val,
        "Canal_Max_Power": canal_max_power,
        "QC_Status": "OK",
    }

    registros_globales.append(fila)

# Convertir a DataFrame global
df_global = pd.DataFrame(registros_globales)

# --- GUARDAR EN EXCEL CON HOJA GENERAL + PESTAÑAS POR GRUPO ---
ruta_excel_global = carpeta_salida_global / "resumen_global_sesiones.xlsx"

with pd.ExcelWriter(ruta_excel_global, engine="openpyxl") as writer:
    # Hoja 1: Todas las sesiones juntas
    df_global.to_excel(writer, sheet_name="TODOS", index=False)

    # Hojas adicionales: Separadas por grupo (WT, NPC, EFV...)
    grupos_unicos = df_global["Grupo"].unique()
    for g in grupos_unicos:
        df_grupo = df_global[df_global["Grupo"] == g]
        nombre_hoja = str(g)[:31]
        df_grupo.to_excel(writer, sheet_name=nombre_hoja, index=False)

print("=" * 65)
print(
    f"¡Éxito! Excel consolidado creado con {len(df_global)} sesiones procesadas."
)
print(f"Ubicación: {ruta_excel_global.resolve()}")
print("=" * 65)

Compilando resumen global de todas las sesiones válidas en: D:\Violeta\ANALISIS\ProcesamientoNatalia_antiguo...

¡Éxito! Excel consolidado creado con 92 sesiones procesadas.
Ubicación: D:\Violeta\ANALISIS\ProcesamientoNatalia_antiguo\resumen_global_sesiones.xlsx
